In [3]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from typing import TypedDict, List, Annotated, Literal
import operator
from langchain_core.prompts import ChatPromptTemplate
from langgraph.types import Send
from langgraph.graph import  StateGraph, START, END
from langchain_groq import ChatGroq
from langchain_ollama import ChatOllama

load_dotenv()

True

In [ ]:
class Task(BaseModel):
    id: int
    title: str
    goal: str = Field(
        ...,
        description="One sentence describing what the reader should be able to do/understand after this section.",
    )
    bullets: List[str] = Field(
        ...,
        min_length=3,
        max_length=5,
        description="3-5 concrete, non-overlapping subpoints to cover in this section.",
    )
    target_words: int = Field(
        ...,
        description="Target word count for this section (120-450).",
    )
    section_type: Literal[
        "intro", "core", "examples", "checklist", "common_mistakes", "conclusion"
    ] = Field(
        ...,
        description="Use 'common_mistakes' and 'examples' exactly once in the plan.",
    )
class Plan(BaseModel):
    blog_title: str = Field(..., description="A catchy and highly technical title for the blog post.")
    audience: str = Field(..., description="Who this blog is for.")
    tone: str = Field(..., description="Writing tone (e.g., practical, crisp).")
    tasks: List[Task]


class RouterDecision(BaseModel):
    need_research: bool
    mode: Literal["closed_book", "hybrid", "open_book"]
    queries: List[str] = Field(default_factory=list)




class State(TypedDict):
    topic: str
    needs_research: bool
    mode: str



    plan: Plan
    sections: Annotated[List[str], operator.add]
    final: str


In [4]:
def get_primary_llm():
    return ChatGoogleGenerativeAI(model='models/gemini-2.5-flash')

def get_secondary_llm():
    # return ChatGroq(model="llama-3.1-8b-instant")
    return ChatOllama(model="llama3")

In [ ]:
def router_node(state : State):
    topic = state["topic"]

    system_prompt = ""

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "")
    ])

    chain = prompt | get_secondary_llm().with_structured_output(RouterDecision)

    
